In [4]:
"""Dataset scan report.

Walks the dataset root, finds every endpoint (leaf) folder that actually
contains files, counts files per extension, and renders a table:

    relative/path/to/leaf   jpg (20), zip (2)
"""

from collections import Counter
from pathlib import Path

import pandas as pd

DATASET_ROOT = Path("/home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing")
REPORT_NAME = "/home/taiaburrahman/dataset_manager_pro/notebook/reports/datasets/v11-processing/"  # used as the prefix in the displayed path


GB = 1024 ** 3


def with_totals(df: pd.DataFrame, label_col: str) -> pd.DataFrame:
    """Append a TOTAL row summing numeric columns."""
    if df.empty:
        return df
    total_row = {col: "" for col in df.columns}
    total_row[label_col] = "TOTAL"
    total_row["Total Files"] = int(df["Total Files"].sum())
    total_row["Size (GB)"] = round(float(df["Size (GB)"].sum()), 3)
    return pd.concat([df, pd.DataFrame([total_row])], ignore_index=True)


def scan_dataset(root: Path) -> pd.DataFrame:
    rows = []
    for folder in root.rglob("*"):
        if not folder.is_dir():
            continue

        files = [p for p in folder.iterdir() if p.is_file()]
        if not files:
            continue  # only keep endpoint (leaf w.r.t. files) folders

        ext_counts: Counter[str] = Counter()
        total_bytes = 0
        for f in files:
            ext = f.suffix.lower().lstrip(".") or "(no_ext)"
            ext_counts[ext] += 1
            try:
                total_bytes += f.stat().st_size
            except OSError:
                pass

        files_str = ", ".join(
            f"{ext} ({count})"
            for ext, count in sorted(ext_counts.items(), key=lambda kv: (-kv[1], kv[0]))
        )

        rel = folder.relative_to(root.parent)  # keep the dataset name as prefix
        rows.append(
            {
                "Path": str(rel),
                "Total Files": sum(ext_counts.values()),
                "Size (GB)": round(total_bytes / GB, 3),
                "Files": files_str,
            }
        )

    df = pd.DataFrame(rows).sort_values("Path").reset_index(drop=True)
    return df


report = with_totals(scan_dataset(DATASET_ROOT), "Path")

# Exclude the TOTAL row from the count summary
data_rows = report[report["Path"] != "TOTAL"] if not report.empty else report
print(f"Dataset root : {DATASET_ROOT}")
print(f"Endpoints    : {len(data_rows)}")
print(f"Total files  : {int(data_rows['Total Files'].sum()) if not data_rows.empty else 0}")
print(f"Total size   : {data_rows['Size (GB)'].sum() if not data_rows.empty else 0:.2f} GB")
print()

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

Dataset root : /home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing
Endpoints    : 477
Total files  : 869031
Total size   : 179.59 GB



In [5]:
out_dir = Path.cwd() / "reports"
out_dir.mkdir(exist_ok=True)

csv_path = out_dir / f"{REPORT_NAME}_scan.csv"
md_path = out_dir / f"{REPORT_NAME}_scan.md"

report.to_csv(csv_path, index=False)
md_path.write_text(report.to_markdown(index=False))

print(f"Saved CSV : {csv_path}")
print(f"Saved MD  : {md_path}")

Saved CSV : /home/taiaburrahman/dataset_manager_pro/notebook/reports/datasets/v11-processing/_scan.csv
Saved MD  : /home/taiaburrahman/dataset_manager_pro/notebook/reports/datasets/v11-processing/_scan.md


In [6]:
"""Main-folder summary.

Aggregates file counts by extension up to MAX_DEPTH path components, e.g.

    v11-processing/fake/<folder>/   jpg (12345), zip (3)

Anything deeper than MAX_DEPTH is rolled up into its 3-step ancestor.
"""

from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

MAX_DEPTH = 3  # e.g. v11-processing / fake / <folder>


def scan_main_folders(root: Path, max_depth: int = MAX_DEPTH) -> pd.DataFrame:
    groups: dict[str, Counter[str]] = defaultdict(Counter)
    group_bytes: dict[str, int] = defaultdict(int)

    base = root.parent  # so the dataset name (v11-processing) is part 0

    for f in root.rglob("*"):
        if not f.is_file():
            continue
        rel_parts = f.relative_to(base).parts
        # truncate to max_depth folder steps; if file is shallower keep its parent
        group_parts = rel_parts[:max_depth]
        # if the file itself was within the depth limit, drop the filename
        if len(rel_parts) <= max_depth:
            group_parts = rel_parts[:-1]
        if not group_parts:
            continue

        group_key = "/".join(group_parts) + "/"
        ext = f.suffix.lower().lstrip(".") or "(no_ext)"
        groups[group_key][ext] += 1
        try:
            group_bytes[group_key] += f.stat().st_size
        except OSError:
            pass

    rows = []
    for group_key, ext_counts in groups.items():
        files_str = ", ".join(
            f"{ext} ({count})"
            for ext, count in sorted(ext_counts.items(), key=lambda kv: (-kv[1], kv[0]))
        )
        rows.append(
            {
                "Folder": group_key,
                "Total Files": sum(ext_counts.values()),
                "Size (GB)": round(group_bytes[group_key] / GB, 3),
                "Files": files_str,
            }
        )

    return pd.DataFrame(rows).sort_values("Folder").reset_index(drop=True)


main_report = with_totals(scan_main_folders(DATASET_ROOT, MAX_DEPTH), "Folder")

data_rows = main_report[main_report["Folder"] != "TOTAL"] if not main_report.empty else main_report
print(f"Dataset root : {DATASET_ROOT}")
print(f"Depth        : {MAX_DEPTH}")
print(f"Main folders : {len(data_rows)}")
print(f"Total files  : {int(data_rows['Total Files'].sum()) if not data_rows.empty else 0}")
print(f"Total size   : {data_rows['Size (GB)'].sum() if not data_rows.empty else 0:.2f} GB")
print()

main_csv = out_dir / f"{REPORT_NAME}_main_folders.csv"
main_md = out_dir / f"{REPORT_NAME}_main_folders.md"
main_report.to_csv(main_csv, index=False)
main_md.write_text(main_report.to_markdown(index=False))
print(f"Saved CSV : {main_csv}")
print(f"Saved MD  : {main_md}")
print()


Dataset root : /home/taiaburrahman/dataset_manager_pro/data/processed/v11-processing
Depth        : 3
Main folders : 20
Total files  : 869031
Total size   : 179.59 GB

Saved CSV : /home/taiaburrahman/dataset_manager_pro/notebook/reports/datasets/v11-processing/_main_folders.csv
Saved MD  : /home/taiaburrahman/dataset_manager_pro/notebook/reports/datasets/v11-processing/_main_folders.md

